In [ ]:
import numpy as np
import cvxpy as cp
import networkx as nx

In [ ]:
# Generate random directed graph (see Erdos–Renyi graph).
num_vertices = 10
connection_prob = .3 # probability with which two vertices are connected by an edge
G = nx.gnp_random_graph(num_vertices, connection_prob, seed=0, directed=True)

# Assign random weights to edges.
min_weight = 1
max_weight = 2
np.random.seed(0)
for u, v in G.edges():
    G[u][v]['weight'] = np.random.uniform(min_weight, max_weight)

# Plot graph.
pos = nx.spring_layout(G)
weights = nx.get_edge_attributes(G, 'weight')
labels = {k: np.round(v, 2) for k, v in weights.items()}
nx.draw(G, pos, with_labels=True)
nx.draw_networkx_edge_labels(G, pos, edge_labels=labels);

In [ ]:
# Source and target vertices.
s = 0
t = num_vertices - 1

# Variables.
num_edges = len(G.edges)
x = cp.Variable(num_edges, boolean=True)
edge_to_var_index = {e: v for v, e in enumerate(G.edges)}

# Constraints.
constraints = []
for v in range(num_vertices):
    delta_sv = int(v == s)
    delta_tv = int(v == t)
    in_sum = sum(x[edge_to_var_index[e]] for e in G.in_edges(v))
    out_sum = sum(x[edge_to_var_index[e]] for e in G.out_edges(v))
    constraints.append(in_sum + delta_sv == out_sum + delta_tv)

# Objective function.
obj = sum(weights[e] * x[edge_to_var_index[e]] for e in G.edges)

# Solve problem.
prob = cp.Problem(cp.Minimize(obj), constraints)
prob.solve()

In [ ]:
# Plot solution.
edge_colors = ['green' if x.value[edge_to_var_index[e]] else 'red' for e in G.edges()]
nx.draw(G, pos, with_labels=True, edge_color=edge_colors)
nx.draw_networkx_edge_labels(G, pos, edge_labels=labels);